# Function-Logit Substrate Annotation Workflow

This notebook is a Colab-friendly front end for the current training workflow.

It is designed to run the scripts in this folder:

- `train_lr_baseline.py`
- `train_mlp_baseline.py`
- `soft_func_logit_decoder.py`

Expected data layout:

```text
code_and_data/
  function_logit_workflow_colab.ipynb
  utils.py
  train_lr_baseline.py
  train_mlp_baseline.py
  soft_func_logit_decoder.py
  BAHD_dataset/
  UGT_dataset/
```


In [ ]:
# Optional: mount Google Drive in Colab
# Uncomment if your ML folder is stored in Drive.

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Setup

Set the path to the folder that contains the scripts and datasets.

In Colab, this is often a Google Drive path.

In [ ]:
from pathlib import Path
import os
import json
import subprocess
from datetime import datetime
from IPython.display import Image
import sys
import pandas as pd
import random
import numpy as np
import importlib
import torch

# Example:
# ML_DIR = Path('/content/drive/MyDrive/your_project/ML')

# Add CS_5782_Final_Project as a shortcut
# to your /MyDrive folder so we can all
# use the same path below

ML_DIR = Path('/content/drive/MyDrive/CS_5782_Final_Project/code_and_data')

assert ML_DIR.exists(), f'ML_DIR does not exist: {ML_DIR}'
os.chdir(ML_DIR)
print('Working directory:', ML_DIR.resolve())


Working directory: /content/drive/MyDrive/CS_5782_Final_Project/code_and_data


In [ ]:
#DEFINE RUN HELPER FUNCTION
def run_cmd(cmd):
    print('\n$', ' '.join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=ML_DIR,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')


In [ ]:
#CHECK IF DATA IS PRESENT
print('Files present:')
for name in [
    'utils.py',
    'train_lr_baseline.py',
    'train_mlp_baseline.py',
    'soft_func_logit_decoder.py',
    'BAHD_dataset',
    'UGT_dataset',
]:
    path = ML_DIR / name
    print(f'  {name}:', 'OK' if path.exists() else 'MISSING')


Files present:
  utils.py: OK
  train_lr_baseline.py: OK
  train_mlp_baseline.py: OK
  soft_func_logit_decoder.py: OK
  BAHD_dataset: OK
  UGT_dataset: OK


In [ ]:
#VERIFY PACKAGES
required = ['numpy', 'torch', 'sklearn']

for pkg in required:
    importlib.import_module(pkg)
    print(f'{pkg}: OK')


numpy: OK
torch: OK
sklearn: OK


In [ ]:
#CHECK CUDA AND GPU
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    print('CUDA device 0:', torch.cuda.get_device_name(0))


CUDA available: False


## 2. Common Run Settings

Change these values once, then reuse them for all runs.

In [ ]:
DATASET = 'BAHD'          # BAHD, UGT, or both
CV_MODE = 'kfold'         # kfold or loocv
N_SPLITS = 5              # used when CV_MODE == 'kfold'
DEVICE = 'auto'           # auto, cpu, cuda, cuda:0
MIN_POSITIVE_COUNT = 2
WRITE_PREDICTIONS = True
SAVE_MODELS = True
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
N_RUNS = 15              #Random search runs
RS_SEED = 42             #Random sampling seed for reproducability

print({
    'dataset': DATASET,
    'cv_mode': CV_MODE,
    'n_splits': N_SPLITS,
    'device': DEVICE,
    'min_positive_count': MIN_POSITIVE_COUNT,
    'write_predictions': WRITE_PREDICTIONS,
    'save_models': SAVE_MODELS,
    'run_tag': RUN_TAG,
})


{'dataset': 'BAHD', 'cv_mode': 'kfold', 'n_splits': 5, 'device': 'auto', 'min_positive_count': 2, 'write_predictions': True, 'save_models': True, 'run_tag': '20260501_123513'}


In [ ]:
#helper function
def make_run_tag(model_name, run_idx):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return f'{model_name}_run{run_idx:02d}_{ts}'

## 3. Exploratory - Logistic Regression Baseline (FL to Label)

In [ ]:
run_cmd([
    'python', 'train_lr_baseline.py',
    '--dataset', DATASET,
    '--cv_mode', CV_MODE,
    '--n_splits', str(N_SPLITS),
    '--device', DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--run_name', f'lr_{RUN_TAG}',
    '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
    '--pickle_models' if SAVE_MODELS else '--no-pickle_models',
])



$ python train_lr_baseline.py --dataset UGT --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --run_name lr_20260427_201944 --write_predictions --pickle_models


## 4. Exploratory - MLP Baseline (FL to Label)

In [ ]:
MLP_POOLING = 'mean'      # mean or attention
MLP_HIDDEN_DIM = 256
MLP_DROPOUT = 0.2
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_EPOCHS = 100
MLP_BATCH_SIZE = 16


In [ ]:
run_cmd([
    'python', 'train_mlp_baseline.py',
    '--dataset', DATASET,
    '--cv_mode', CV_MODE,
    '--n_splits', str(N_SPLITS),
    '--device', DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--pooling', MLP_POOLING,
    '--hidden_dim', str(MLP_HIDDEN_DIM),
    '--dropout', str(MLP_DROPOUT),
    '--learning_rate', str(MLP_LR),
    '--weight_decay', str(MLP_WEIGHT_DECAY),
    '--epochs', str(MLP_EPOCHS),
    '--batch_size', str(MLP_BATCH_SIZE),
    '--run_name', f'mlp_{MLP_POOLING}_{RUN_TAG}',
    '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
    '--save_models' if SAVE_MODELS else '--no-save_models',
])



$ python train_mlp_baseline.py --dataset UGT --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --pooling mean --hidden_dim 256 --dropout 0.2 --learning_rate 0.001 --weight_decay 0.0001 --epochs 100 --batch_size 16 --run_name mlp_mean_20260427_201944 --write_predictions --save_models
/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/train_mlp_baseline.py:274: RuntimeWarning: overflow encountered in exp
  preds = (1.0 / (1.0 + np.exp(-logits)) >= 0.5).astype(np.uint8, copy=False)



## 5. Exploratory - Soft Function-Logit Decoder (FL to Label)

In [ ]:
SOFT_POOLING = 'attention'   # mean or attention
SOFT_D_MODEL = 256
SOFT_N_HEADS = 4
SOFT_N_LAYERS = 2
SOFT_DROPOUT = 0.1
SOFT_TEMPERATURE = 1.0
SOFT_LR = 1e-4
SOFT_WEIGHT_DECAY = 1e-4
SOFT_EPOCHS = 60
SOFT_BATCH_SIZE = 8


In [ ]:
run_cmd([
    'python', 'soft_func_logit_decoder.py',
    '--dataset', DATASET,
    '--cv_mode', CV_MODE,
    '--n_splits', str(N_SPLITS),
    '--device', DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--pooling', SOFT_POOLING,
    '--d_model', str(SOFT_D_MODEL),
    '--n_heads', str(SOFT_N_HEADS),
    '--n_layers', str(SOFT_N_LAYERS),
    '--dropout', str(SOFT_DROPOUT),
    '--temperature', str(SOFT_TEMPERATURE),
    '--learning_rate', str(SOFT_LR),
    '--weight_decay', str(SOFT_WEIGHT_DECAY),
    '--epochs', str(SOFT_EPOCHS),
    '--batch_size', str(SOFT_BATCH_SIZE),
    '--run_name', f'soft_decoder_{SOFT_POOLING}_{RUN_TAG}',
    '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
    '--save_models' if SAVE_MODELS else '--no-save_models',
])


## 6. Summarize Exploratory Metrics (old)

This scans the `results/` folder and prints the main metrics from every run.

In [ ]:
# def collect_metric_files(results_root: Path):
#     return sorted(results_root.glob('**/*_metrics.json'))


# def summarize_metric_file(path: Path):
#     with path.open('r', encoding='utf-8') as handle:
#         payload = json.load(handle)
#     metrics = payload.get('metrics', {})
#     return {
#         'run_dir': path.parent.name,
#         'file': path.name,
#         'family': payload.get('family'),
#         'task': payload.get('task'),
#         'micro_aupr': metrics.get('micro_aupr'),
#         'micro_auroc': metrics.get('micro_auroc'),
#         'macro_aupr': metrics.get('macro_aupr'),
#         'macro_auroc': metrics.get('macro_auroc'),
#     }


# metric_files = collect_metric_files(ML_DIR / 'results')
# print('Metric files found:', len(metric_files))
# for path in metric_files:
#     print(summarize_metric_file(path))

In [ ]:
def collect_metric_files(results_root: Path):
    return sorted(results_root.glob('**/*_metrics.json'))


def summarize_metric_file(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    metrics = payload.get('metrics', {})
    logistic_params = payload.get("logistic_params", {})
    mlp_params = payload.get("mlp_params", {})
    decoder_params = payload.get("decoder_params", {})
    return {
        'run_dir': path.parent.name,
        'file': path.name,
        'family': payload.get('family'),
        'task': payload.get('task'),
        "pooling": (
            payload.get("pooling")
            or mlp_params.get("pooling")
            or decoder_params.get("pooling")
        ),
        "learning_rate": (
            mlp_params.get("learning_rate")
            or decoder_params.get("learning_rate")
        ),
        "dropout": (
            mlp_params.get("dropout")
            or decoder_params.get("dropout")
        ),
        "model_type": path.parent.parent.name,
        'micro_aupr': metrics.get('micro_aupr'),
        'micro_auroc': metrics.get('micro_auroc'),
        'macro_aupr': metrics.get('macro_aupr'),
        'macro_auroc': metrics.get('macro_auroc'),
    }


metric_files = collect_metric_files(ML_DIR / 'results')
print('Metric files found:', len(metric_files))
for path in metric_files:
    print(summarize_metric_file(path))


#Extract the info
summary_rows = [summarize_metric_file(path) for path in metric_files]
metric_summary_df = pd.DataFrame(summary_rows)
metric_summary_df

Metric files found: 27
{'run_dir': 'embedding_transformer_attention_20260427_151725', 'file': 'bahd_acceptor_metrics.json', 'family': 'BAHD', 'task': 'acceptor', 'pooling': 'attention', 'learning_rate': None, 'dropout': None, 'model_type': 'embedding_transformer', 'micro_aupr': 0.6180435736877133, 'micro_auroc': 0.884750245149021, 'macro_aupr': 0.48889713963967035, 'macro_auroc': 0.8213347812928468}
{'run_dir': 'embedding_transformer_attention_20260427_151725', 'file': 'bahd_donor_metrics.json', 'family': 'BAHD', 'task': 'donor', 'pooling': 'attention', 'learning_rate': None, 'dropout': None, 'model_type': 'embedding_transformer', 'micro_aupr': 0.9338228494660413, 'micro_auroc': 0.940740380004795, 'macro_aupr': 0.9336119189564984, 'macro_auroc': 0.9429314113702736}
{'run_dir': 'lr_20260423_173500', 'file': 'bahd_acceptor_metrics.json', 'family': 'BAHD', 'task': 'acceptor', 'pooling': None, 'learning_rate': None, 'dropout': None, 'model_type': 'lr_baseline', 'micro_aupr': 0.475146840031

,run_dir,file,family,task,pooling,learning_rate,dropout,model_type,micro_aupr,micro_auroc,macro_aupr,macro_auroc
0,embedding_transformer_attention_20260427_151725,bahd_acceptor_metrics.json,BAHD,acceptor,attention,NaN,NaN,embedding_transformer,0.618044,0.884750,0.488897,0.821335
1,embedding_transformer_attention_20260427_151725,bahd_donor_metrics.json,BAHD,donor,attention,NaN,NaN,embedding_transformer,0.933823,0.940740,0.933612,0.942931
2,lr_20260423_173500,bahd_acceptor_metrics.json,BAHD,acceptor,None,NaN,NaN,lr_baseline,0.475147,0.886577,0.394111,0.811691
3,lr_20260423_173500,bahd_donor_metrics.json,BAHD,donor,None,NaN,NaN,lr_baseline,0.927519,0.920849,0.922535,0.920260
4,lr_20260424_141104,bahd_acceptor_metrics.json,BAHD,acceptor,None,NaN,NaN,lr_baseline,0.475147,0.886577,0.394111,0.811691
5,lr_20260424_141104,bahd_donor_metrics.json,BAHD,donor,None,NaN,NaN,lr_baseline,0.927519,0.920849,0.922535,0.920260
6,lr_20260424_144205,bahd_acceptor_metrics.json,BAHD,acceptor,None,NaN,NaN,lr_baseline,0.475147,0.886577,0.394111,0.811691
7,lr_20260424_144205,bahd_donor_metrics.json,BAHD,donor,None,NaN,NaN,lr_baseline,0.927519,0.920849,0.922535,0.920260
8,lr_20260427_201944,ugt_acceptor_metrics.json,UGT,acceptor,None,NaN,NaN,lr_baseline,0.182466,0.821840,0.177685,0.766071
9,lr_20260427_201944,ugt_donor_metrics.json,UGT,donor,None,NaN,NaN,lr_baseline,0.278670,0.846669,0.162034,0.598844


##7: Random Search - Logistic Regression

In [ ]:
rng = random.Random(RS_SEED)
np_rng = np.random.default_rng(RS_SEED)

fl_lr_tags = []

for i in range(N_RUNS):
    C       = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(100)))
    penalty = rng.choice(['l1', 'l2'])
    tag     = make_run_tag('lr_baseline', i)
    fl_lr_tags.append(tag)

    print(f'\n--- FL-LR run {i:02d} | C={C:.5f} penalty={penalty} ---')
    run_cmd([
        'python', 'train_lr_baseline.py',
        '--dataset',            DATASET,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--C',                  f'{C:.8f}',
        '--penalty',            penalty,
        '--run_name',           tag,
        '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
        '--no-pickle_models',
    ])
    try:
        m_path = ML_DIR / 'results' / 'lr_baseline' / tag / 'bahd_acceptor_metrics.json'
        with open(m_path) as f:
            m = json.load(f)
        print(f'  Run {i:02d} done | acceptor micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nFL-LR random search complete.')


--- FL-LR run 00 | C=4.40287 penalty=l1 ---

$ python train_lr_baseline.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --C 4.40287435 --penalty l1 --run_name lr_baseline_run00_20260428_223341 --write_predictions --no-pickle_models
  Run 00 done | acceptor micro_aupr = 0.5392

--- FL-LR run 01 | C=0.04298 penalty=l1 ---

$ python train_lr_baseline.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --C 0.04298042 --penalty l1 --run_name lr_baseline_run01_20260428_223603 --write_predictions --no-pickle_models
  Run 01 done | acceptor micro_aupr = 0.1712

--- FL-LR run 02 | C=14.17710 penalty=l2 ---

$ python train_lr_baseline.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --C 14.17710381 --penalty l2 --run_name lr_baseline_run02_20260428_223622 --write_predictions --no-pickle_models
  Run 02 done | acceptor micro_aupr = 0.5826

--- FL-LR run 03 | C=1.52830 penalty=l1 ---

$ python t

##8: Random Search - Multi Layer Perceptron

In [ ]:
rng    = random.Random(RS_SEED)
np_rng = np.random.default_rng(RS_SEED)
fl_mlp_tags = []

#Variables to random search
VALID_HIDDEN_DIMS   = [64, 128, 256, 512]
VALID_DROPOUTS      = [0.0, 0.1, 0.2, 0.3, 0.5]
VALID_WEIGHT_DECAYS = [0, 1e-5, 1e-4, 1e-3]
VALID_POOLINGS      = ['mean', 'attention']

for i in range(N_RUNS):
    hidden_dim   = rng.choice(VALID_HIDDEN_DIMS)
    dropout      = rng.choice(VALID_DROPOUTS)
    lr           = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(1e-2)))
    weight_decay = rng.choice(VALID_WEIGHT_DECAYS)
    pooling      = rng.choice(VALID_POOLINGS)
    tag          = make_run_tag('mlp_baseline', i)
    fl_mlp_tags.append(tag)

    print(f'\n--- FL-MLP run {i:02d} | hidden={hidden_dim} drop={dropout} lr={lr:.5f} wd={weight_decay} pool={pooling} ---')
    run_cmd([
        'python', 'train_mlp_baseline.py',
        '--dataset',                 DATASET,
        '--cv_mode',                 CV_MODE,
        '--n_splits',                str(N_SPLITS),
        '--device',                  DEVICE,
        '--min_positive_count',      str(MIN_POSITIVE_COUNT),
        '--hidden_dim',              str(hidden_dim),
        '--dropout',                 str(dropout),
        '--learning_rate',           f'{lr:.8f}',
        '--weight_decay',            str(weight_decay),
        '--pooling',                 pooling,
        '--epochs',                  '500',
        '--batch_size',              '16',
        '--early_stopping_patience', '10',
        '--run_name',                tag,
        '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
        '--no-save_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'mlp_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nFL-MLP random search complete.')


--- FL-MLP run 00 | hidden=64 drop=0.0 lr=0.00353 wd=0.0001 pool=mean ---

$ python train_mlp_baseline.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --hidden_dim 64 --dropout 0.0 --learning_rate 0.00353112 --weight_decay 0.0001 --pooling mean --epochs 500 --batch_size 16 --early_stopping_patience 10 --run_name mlp_baseline_run00_20260429_001255 --write_predictions --no-save_models
  Run 00 done | donor micro_aupr = 0.8620
  Run 00 done | acceptor micro_aupr = 0.0993

--- FL-MLP run 01 | hidden=128 drop=0.1 lr=0.00075 wd=0 pool=mean ---

$ python train_mlp_baseline.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --hidden_dim 128 --dropout 0.1 --learning_rate 0.00075467 --weight_decay 0 --pooling mean --epochs 500 --batch_size 16 --early_stopping_patience 10 --run_name mlp_baseline_run01_20260429_002115 --write_predictions --no-save_models
  Run 01 done | donor micro_aupr = 0.8612
  Run 01 done | acceptor micro_au

##9: Random Search - Soft Function Logit Decoder

In [ ]:
rng    = random.Random(RS_SEED)
np_rng = np.random.default_rng(RS_SEED)
fl_sfd_tags = []

#Variables to random search
VALID_D_NHEAD_PAIRS = [(64,4),(64,8),(128,4),(128,8),(256,4),(256,8)]
VALID_TEMPERATURES  = [0.5, 1.0, 2.0, 5.0]
VALID_NUM_LAYERS    = [1, 2, 3]
VALID_DROPOUTS      = [0.0, 0.1, 0.2, 0.3]
VALID_POOLINGS      = ['mean', 'attention']

for i in range(N_RUNS):
    d_model, n_heads = rng.choice(VALID_D_NHEAD_PAIRS)
    temperature      = rng.choice(VALID_TEMPERATURES)
    n_layers         = rng.choice(VALID_NUM_LAYERS)
    dropout          = rng.choice(VALID_DROPOUTS)
    pooling          = rng.choice(VALID_POOLINGS)
    lr               = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(5e-3)))
    tag              = make_run_tag('soft_func_logit_decoder', i)
    fl_sfd_tags.append(tag)

    print(f'\n--- FL-SFD run {i:02d} | d_model={d_model} n_heads={n_heads} layers={n_layers} drop={dropout} temp={temperature} pool={pooling} lr={lr:.5f} ---')
    run_cmd([
        'python', 'soft_func_logit_decoder.py',
        '--dataset',            DATASET,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--d_model',            str(d_model),
        '--n_heads',            str(n_heads),
        '--n_layers',           str(n_layers),
        '--dropout',            str(dropout),
        '--pooling',            pooling,
        '--temperature',        str(temperature),
        '--learning_rate',      f'{lr:.8f}',
        '--epochs',             '100',
        '--batch_size',         '8',
        '--run_name',           tag,
        '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
        '--no-save_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'soft_func_logit_decoder' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nFL-SFD random search complete.')


--- FL-SFD run 00 | d_model=256 n_heads=8 layers=1 drop=0.2 temp=0.5 pool=mean lr=0.00207 ---

$ python soft_func_logit_decoder.py --dataset BAHD --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --d_model 256 --n_heads 8 --n_layers 1 --dropout 0.2 --pooling mean --temperature 0.5 --learning_rate 0.00206504 --epochs 100 --batch_size 8 --run_name soft_func_logit_decoder_run00_20260429_143119 --write_predictions --no-save_models
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(

  Run 00 done | donor micro_aupr = 0.9142
  Run 00 done | acceptor m

##10: Homebrewed LazyPredict Screening

In [ ]:
#LOAD FEATURES
import time
from pathlib import Path
from utils import build_sequence_feature_list, load_dataset_cache, dataset_dir_for_family

dataset_dir = dataset_dir_for_family("BAHD")
cache = load_dataset_cache(dataset_dir)

print("Loading FL features...")
t0 = time.time()
sequences = build_sequence_feature_list(dataset_dir, cache, device="cpu", flatten=True)
print(f"Done in {time.time()-t0:.1f}s — {len(sequences)} sequences loaded")

Loading FL features...
Done in 3.9s — 366 sequences loaded


In [ ]:
#SCREEN PREDICTIONS
import sys
sys.argv = [
    "classifier_screen.py",
    "--feature_type",       "function_logits",
    "--dataset",            DATASET,
    "--task",               "both",
    "--cv_mode",            CV_MODE,
    "--n_splits",           str(N_SPLITS),
    "--min_positive_count", str(MIN_POSITIVE_COUNT),
    "--random_seed",        str(RS_SEED),
]

import importlib
import classifier_screen
importlib.reload(classifier_screen)
classifier_screen.main()


  BAHD | donor | function_logits
  X shape      : (366, 2080)
  Labels       : 2
  CV splits    : 5

  [LogisticRegression] micro_aupr=0.9367
  [RidgeClassifier] micro_aupr=0.9346
  [LinearSVC] 

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

micro_aupr=0.9337
  [SGDClassifier] 

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


micro_aupr=0.8631
  [RandomForest] micro_aupr=0.9199
  [ExtraTrees] micro_aupr=0.9206
  [DecisionTree] micro_aupr=0.7527
  [HistGradientBoosting] micro_aupr=0.9187
  [KNeighbors] micro_aupr=0.8557
  [GaussianNB] micro_aupr=0.7255
  [BernoulliNB] micro_aupr=0.7214
  [SVC_RBF] micro_aupr=0.8313
  [LinearDiscriminantAnalysis] micro_aupr=0.8729
  [LightGBM] 

...

micro_aupr=0.1458
  [RandomForest] micro_aupr=0.4969
  [ExtraTrees] micro_aupr=0.4564
  [DecisionTree] micro_aupr=0.2204
  [HistGradientBoosting] micro_aupr=0.5033
  [KNeighbors] micro_aupr=0.3412
  [GaussianNB] micro_aupr=0.0910
  [BernoulliNB] micro_aupr=0.0786
  [SVC_RBF] micro_aupr=0.5040
  [LinearDiscriminantAnalysis] micro_aupr=0.3686
  [LightGBM] 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

micro_aupr=0.4769
  [XGBoost] 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

micro_aupr=0.4931

  Rank Classifier                     Micro AUPR Macro AUPR Micro AUROC
  -------------------------------------------------------------------
  1    RidgeClassifier                    0.6532     0.4936      0.8907
  2    LinearSVC                          0.5788     0.4573      0.8782
  3    LogisticRegression                 0.5753     0.4248      0.8982
  4    SVC_RBF                            0.5040     0.3383      0.8553
  5    HistGradientBoosting               0.5033     0.3581      0.8687
  6    RandomForest                       0.4969     0.3534      0.8549
  7    XGBoost                            0.4931     0.3356      0.8348
  8    LightGBM                           0.4769     0.3543      0.8569
  9    ExtraTrees                         0.4564     0.3417      0.8482
  10   LinearDiscriminantAnalysis         0.3686     0.3429      0.8443
  11   KNeighbors                         0.3412     0.2153      0.8082
  12   DecisionTree                       0.220

##11: Random Search - RidgeClassifier

In [ ]:

N_RUNS  = 15
RS_SEED = 42
np_rng  = np.random.default_rng(RS_SEED)
ridge_fl_tags = []

for i in range(N_RUNS):
    alpha = float(10 ** np_rng.uniform(np.log10(1e-3), np.log10(1e3)))
    tag   = make_run_tag('ridge_baseline', i)
    ridge_fl_tags.append(tag)

    print(f'\n--- Ridge (FL) run {i:02d} | alpha={alpha:.5f} ---')
    run_cmd([
        'python', 'train_ridge_baseline.py',
        '--dataset',            DATASET,
        '--alpha',              f'{alpha:.8f}',
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED),
        '--run_name',           tag,
        '--no-pickle_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'ridge_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nRidge (FL) random search complete.')


--- Ridge (FL) run 00 | alpha=44.02874 ---

$ python train_ridge_baseline.py --dataset BAHD --alpha 44.02874347 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 42 --run_name ridge_baseline_run00_20260429_212404 --no-pickle_models
  Run 00 done | donor micro_aupr = 0.9276
  Run 00 done | acceptor micro_aupr = 0.4757

--- Ridge (FL) run 01 | alpha=0.42980 ---

$ python train_ridge_baseline.py --dataset BAHD --alpha 0.42980418 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 42 --run_name ridge_baseline_run01_20260429_212429 --no-pickle_models
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.72523e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.62827e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwarg

##12: Large Random Search on Best FL Model - Ridge Classifier

In [ ]:
N_RUNS_LARGE  = 30
RS_SEED_LARGE = 43
np_rng_large  = np.random.default_rng(RS_SEED_LARGE)
ridge_large_tags = []

for i in range(N_RUNS_LARGE):
    alpha = float(10 ** np_rng_large.uniform(np.log10(1e-3), np.log10(1e3)))
    tag   = make_run_tag('ridge_baseline_large', i)
    ridge_large_tags.append(tag)

    print(f'\n--- Large run {i:03d} | alpha={alpha:.5f} ---')
    run_cmd([
        'python', 'train_ridge_baseline.py',
        '--dataset',            DATASET,
        '--alpha',              f'{alpha:.8f}',
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED_LARGE),
        '--run_name',           tag,
        '--no-pickle_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'ridge_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:03d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:03d} done (no metrics found)')

print('\nPhase 1 complete.')
print(ridge_large_tags)


--- Large run 000 | alpha=8.19965 ---

$ python train_ridge_baseline.py --dataset BAHD --alpha 8.19965493 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 43 --run_name ridge_baseline_large_run00_20260501_011715 --no-pickle_models


KeyboardInterrupt: 

## 13: Refined Random Search on best FL Model - Ridge Classifier

In [ ]:
#chosen range: 0.1-1
LOW_REFINED = 1e-1
HIGH_REFINED = 1e0
N_RUNS_REFINED  = 30
RS_SEED_REFINED = 44
np_rng_refined  = np.random.default_rng(RS_SEED_REFINED)
ridge_refined_tags = []

for i in range(N_RUNS_REFINED):
    alpha = float(10 ** np_rng_refined.uniform(np.log10(LOW_REFINED), np.log10(HIGH_REFINED))) #PUT CHOSEN RANGE HERE
    tag   = make_run_tag('ridge_baseline_refined', i)
    ridge_refined_tags.append(tag)

    print(f'\n--- Refined run {i:03d} | alpha={alpha:.5f} ---')
    run_cmd([
        'python', 'train_ridge_baseline.py',
        '--dataset',            DATASET,
        '--alpha',              f'{alpha:.8f}',
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED_REFINED),
        '--run_name',           tag,
        '--no-pickle_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'ridge_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:03d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:03d} done (no metrics found)')

print('\nRefined search complete.')
print(ridge_refined_tags)

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.02092e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.62673e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.07017e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.4865e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.94564e-08): result may not be accurate.
  return f(*a

##14: Final FL

In [ ]:
#WINNING HYPERPARAMETER
BEST_ALPHA = 0.47612701  # refined run08

In [ ]:
#FINAL RUN - 5 FOLD CV
FINAL_RIDGE_5FOLD_TAG = make_run_tag('ridge_final_5fold', 0)
print(f'Running 5-fold CV | alpha={BEST_ALPHA} | tag={FINAL_RIDGE_5FOLD_TAG}')
run_cmd([
    'python', 'train_ridge_baseline.py',
    '--dataset',            DATASET,
    '--alpha',              f'{BEST_ALPHA:.8f}',
    '--cv_mode',            'kfold',
    '--n_splits',           str(N_SPLITS),
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_RIDGE_5FOLD_TAG,
    '--no-pickle_models',
    '--write_predictions',
])
for task in ['donor', 'acceptor']:
    m_path = ML_DIR / 'results' / 'ridge_baseline' / FINAL_RIDGE_5FOLD_TAG / f'bahd_{task}_metrics.json'
    with open(m_path) as f:
        m = json.load(f)
    print(f'  5-fold | {task:8s} | micro_aupr={m["metrics"]["micro_aupr"]:.4f}  macro_aupr={m["metrics"]["macro_aupr"]:.4f}  micro_auroc={m["metrics"]["micro_auroc"]:.4f}  macro_auroc={m["metrics"]["macro_auroc"]:.4f}')

Running 5-fold CV | alpha=0.47612701 | tag=ridge_final_5fold_run00_20260501_144618

$ python train_ridge_baseline.py --dataset BAHD --alpha 0.47612701 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 45 --run_name ridge_final_5fold_run00_20260501_144618 --no-pickle_models --write_predictions
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.9439e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.32779e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.87049e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-cond

In [ ]:
#FINAL RUN - LOOCV
FINAL_RIDGE_LOOCV_TAG = make_run_tag('ridge_final_loocv', 0)
print(f'Running LOOCV | alpha={BEST_ALPHA} | tag={FINAL_RIDGE_LOOCV_TAG}')
run_cmd([
    'python', 'train_ridge_baseline.py',
    '--dataset',            DATASET,
    '--alpha',              f'{BEST_ALPHA:.8f}',
    '--cv_mode',            'loocv',
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_RIDGE_LOOCV_TAG,
    '--no-pickle_models',
    '--write_predictions',
])
for task in ['donor', 'acceptor']:
    m_path = ML_DIR / 'results' / 'ridge_baseline' / FINAL_RIDGE_LOOCV_TAG / f'bahd_{task}_metrics.json'
    with open(m_path) as f:
        m = json.load(f)
    print(f'  LOOCV  | {task:8s} | micro_aupr={m["metrics"]["micro_aupr"]:.4f}  macro_aupr={m["metrics"]["macro_aupr"]:.4f}  micro_auroc={m["metrics"]["micro_auroc"]:.4f}  macro_auroc={m["metrics"]["macro_auroc"]:.4f}')

Streaming output truncated to the last 5000 lines.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.25787e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.60619e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.63152e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.72623e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.43924e-08

In [ ]:
#FINAL RUN FOLD COMPARISON
print(f'{"":12s} {"Micro AUPR":>10} {"Macro AUPR":>10} {"Micro AUROC":>11} {"Macro AUROC":>11}')
print('-' * 57)
for tag, label in [(FINAL_RIDGE_5FOLD_TAG, '5-fold'), (FINAL_RIDGE_LOOCV_TAG, 'LOOCV')]:
    for task in ['donor', 'acceptor']:
        m_path = ML_DIR / 'results' / 'ridge_baseline' / tag / f'bahd_{task}_metrics.json'
        with open(m_path) as f:
            m = json.load(f)['metrics']
        print(f'{label} {task:8s}  {m["micro_aupr"]:>10.4f} {m["macro_aupr"]:>10.4f} {m["micro_auroc"]:>11.4f} {m["macro_auroc"]:>11.4f}')
    print()

             Micro AUPR Macro AUPR Micro AUROC Macro AUROC
---------------------------------------------------------
5-fold donor         0.9424     0.9406      0.9532      0.9535
5-fold acceptor      0.6222     0.5314      0.9128      0.8253

LOOCV donor         0.9487     0.9454      0.9571      0.9566
LOOCV acceptor      0.6364     0.5390      0.9166      0.8074



In [ ]:
FINAL_RIDGE_FULL_TAG = make_run_tag('ridge_final_full', 0)
print(f'Full-data refit | alpha={BEST_ALPHA} | tag={FINAL_RIDGE_FULL_TAG}')
run_cmd([
    'python', 'train_ridge_baseline.py',
    '--dataset',            DATASET,
    '--alpha',              f'{BEST_ALPHA:.8f}',
    '--cv_mode',            'kfold',
    '--n_splits',           str(N_SPLITS),
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_RIDGE_FULL_TAG,
    '--pickle_models',
    '--no-write_predictions',
])
print(f'Model saved to: results/ridge_baseline/{FINAL_RIDGE_FULL_TAG}/')

Full-data refit | alpha=0.47612701 | tag=ridge_final_full_run00_20260501_123532

$ python train_ridge_baseline.py --dataset BAHD --alpha 0.47612701 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 45 --run_name ridge_final_full_run00_20260501_123532 --pickle_models --no-write_predictions
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.9439e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.32779e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.87049e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditio